# Ingestão de dados para RAG com LLM

Este notebook tem como objetivo realizar o processamento e divisão de arquivos Markdown, organizando os textos extraídos para ingestão em uma vector store. O propósito final é habilitar a utilização desses dados em um agente RAG (Retrieval-Augmented Generation), permitindo consultas eficientes e contextualizadas. 

A abordagem inclui:

- **Configuração do modelo LLM**: Utilizando o modelo `deepseek-r1:8b` para interações e processamento de linguagem natural.
- **Ingestão de dados**: Extração e organização dos textos Markdown para armazenamento estruturado.
- **Integração com vector store**: Preparação dos dados para consultas otimizadas em um agente RAG.

Este fluxo garante que os textos sejam processados de forma eficiente, permitindo a criação de agentes inteligentes baseados em recuperação de informações.

## 0. Configuração do ambiente

In [ ]:
# Configurations

OLLAMA_LLM_MODEL = "deepseek-r1:8b"

OLLAMA_EMBEDDING_MODEL = "mxbai-embed-large:latest"
# OLLAMA_EMBEDDING_MODEL = "nomic-embed-text:latest"

BIBLE_MD_PATH = "../../data/processed/biblia/pdf_to_markdown_using_marker-pdf/Bíblia Sagrada O Antigo e Novo Testamento - 4 volumes - Vulgata Latina por Pe. Matos Soares 1927-1950.md"
CATECISM_MD_PATH = "../../data/processed/catecismo/pdf_to_markdown_using_marker/Catecismo da Igreja Católica.WithoutLLM.md"

# VALIDATION_QUESTION = "Qual é o versículo mais longo da Bíblia?"
VALIDATION_QUESTION = "Mostre as passagens da bíblia que discorrem sobre os vícios. Quais são os vícios da Carne? Como devemos lidar com os vícios?"

In [ ]:
import rich

def rich_print(text: str):
    rich.print(f"""[white]{text}[/white]""")

## 1. Configurando Ollama

In [ ]:
import ollama

installed_models = list(map(lambda model: model.model, ollama.list().models))
installed_models

In [ ]:
if OLLAMA_LLM_MODEL not in installed_models:
    print(f'Model "{OLLAMA_LLM_MODEL}" is not installed. Pulling...')
    ollama.pull(OLLAMA_LLM_MODEL)
else:
    print(f'Model "{OLLAMA_LLM_MODEL}" is already installed.')

if OLLAMA_EMBEDDING_MODEL not in installed_models:
    print(f'Embedding model "{OLLAMA_EMBEDDING_MODEL}" is not installed. Pulling...')
    ollama.pull(OLLAMA_EMBEDDING_MODEL)
else:
    print(f'Embedding model "{OLLAMA_EMBEDDING_MODEL}" is already installed.')

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from pprint import pprint

llm = ChatOllama(
    model=OLLAMA_LLM_MODEL,
    temperature=0.8,
    # num_predict = 256,
    # other params ...
)

messages = [
    SystemMessage(
        """
Você é o Ollama. 
Você é um assistente de IA que fala português fluentemente.
Você é amigável e útil. Você não deve se comportar como um humano, mas sim como uma IA.
Você deve ser sempre educado e respeitoso.
Você deve ser muito bem humorado.
"""
    ),
    HumanMessage("Ollá Ollama! Prove que você é tão bom quanto dizem que você é."),
]

pprint(llm.invoke(messages).content)

## 2. Extraindo texto dos arquivos Markdown

### 2.1. (Apenas demonstração) Extraindo texto dos arquivos Markdown

**Não vamos utilizar esse método.**

Motivo: ao invés, vamos extrair o markdown inteiro (com as marcacoes) e depois fazer o split em cada arquivo markdown. Assim, conseguimos manter a formatação original do arquivo markdown.

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document

loader = UnstructuredMarkdownLoader(BIBLE_MD_PATH)

data = loader.load()
assert len(data) == 1
assert isinstance(data[0], Document)
bible_content = data[0].page_content
print(bible_content[:250])

In [ ]:
bible_content[:1000]

### 2.2. Extraindo texto dos arquivos Markdown (formatado como markdown)

In [ ]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader(BIBLE_MD_PATH, autodetect_encoding=True)

bible_docs = loader.load()

bible_text = bible_docs[0].page_content

print(bible_text[11000:12000])

## 3. Dividindo o texto em partes menores

### 3.1. Bíblia

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Livro ou Capítulo"),
    ("##", "Subcapítulo"),
    ("####", "Subdivisão"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
bible_header_splits = markdown_splitter.split_text(bible_text)

pprint(bible_header_splits[600].metadata)

print("\n")
pprint(bible_header_splits[600].page_content)

### 3.2. Catecismo da Igreja Católica

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(separator="\n\n")

catecism_loader = TextLoader(CATECISM_MD_PATH, autodetect_encoding=True)
catecism_docs = catecism_loader.load()

catecism_text = catecism_docs[0].page_content

headers_to_split_on = [
    ("#", "Capítulo"),
    ("####", "Subcapítulo"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
catecism_header_splits = markdown_splitter.split_text(catecism_text)

pprint(catecism_header_splits[200].metadata)

print("\n")
pprint(catecism_header_splits[200].page_content)

## 4. Criando embeddings para os textos

In [ ]:
from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(model=OLLAMA_EMBEDDING_MODEL)

# Testing the embedding model
ollama_embeddings.embed_query("Hello, world!")[:10]

In [ ]:
# Não vamos utilizar
# Motivo: será utilizado internamente na vector store

# from tqdm import tqdm

# embeddings = []
# for split in tqdm(bible_header_splits, desc="Generating embeddings"):
#     embeddings.append(ollama_embeddings.embed_query(split.page_content))
    
# embeddings[0][:10]


## 5. Criando a vector store

### 5.1. Criando a vector store

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=ollama_embeddings)

### 5.2. Adicionando textos da Bíblia à vector store

In [ ]:
_ = vector_store.add_documents(bible_header_splits)

### 5.3. Adicionando textos do Catecismo à vector store

In [ ]:
_ = vector_store.add_documents(catecism_header_splits)

## 6. Criando o agente RAG

[🔗 From](https://python.langchain.com/docs/tutorials/rag/#orchestration)

In [ ]:
# Example Prompt Template
# from langchain import hub

# prompt = hub.pull("rlm/rag-prompt")

# example_messages = prompt.invoke(
#     {"context": "(context goes here)", "question": "(question goes here)"}
# ).to_messages()

# assert len(example_messages) == 1
# print(example_messages[0].content)

In [ ]:
# Our prompt template (which includes the system message)

from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_core.prompts import SystemMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template(
"""
Você é um assistente de IA especializado em responder perguntas sobre o catolicismo.
Você fala português fluentemente e responde sempre em português brasileiro.
Você entende profundamente a Bíblia, a tradição católica, e a fé católica.
Você conhece os sacramentos, a liturgia, a vida de oração, e o ritual da missa, incluindo a importância da Eucaristia.
Você é capaz de responder perguntas sobre a história da Igreja Católica, seus santos, mártires, papas, bispos, padres, diáconos, leigos, religiosos, consagrados, missionários, catequistas, evangelizadores e profetas.
Você fornece conselhos e orientações sobre questões relacionadas à fé católica e à vida cotidiana, alinhados com a Bíblia e a tradição católica.
Você deve, sempre que possível, citar as passagens bíblicas ou ensinamentos do catecismo da Igreja Católica, mostrando a referência e o versículo.
Sempre que citar uma passagem da Bíblia, você deve transcrever o versículo na íntegra, em discurso direto, entre aspas, e não apenas a referência. Para trechos pequenos, a transcrição deve ser integral e explícita, sem modificações, exceto se o trecho citado for excessivamente grande. Por exemplo: "Porque Deus amou o mundo de tal maneira que deu o seu Filho unigênito, para que todo aquele que nele crê não pereça, mas tenha a vida eterna." (João 3:16).
Você deve ser educado, respeitoso, e basear suas respostas no contexto fornecido, sem inventar informações ou fazer suposições.
Você deve sempre responder sob a ótica católica, e nunca sob a ótica de outras religiões ou filosofias.
Você deve sempre responder de forma clara e concisa, evitando jargões técnicos ou termos complexos.
Ao final da resposta, sempre estimule o usuário a continuar a conversa, fazendo uma pergunta, sugerindo um tópico relacionado ou sugerindo melhorar a explicação de algum dos termos.
"""
        ),
        HumanMessagePromptTemplate.from_template(
"""
Contexto: {context}
Pergunta: {question}
"""
        ),
    ]
)

example_messages = prompt.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

for message in example_messages:
    rich_print(message.content)

### 6.1 . Utilizando LangGraph para monitorar o prompt de entrada, o contexto trazido e a resposta produzida

In [ ]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str


def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"], k=10)
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

In [ ]:
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [ ]:
import mermaid
from mermaid.graph import Graph

# Generate the Mermaid diagram locally
mermaid_code = graph.get_graph().draw_mermaid()
mermaid_code

merma_graph = Graph('Sequence-diagram', mermaid_code)
render = mermaid.Mermaid(
    graph=merma_graph,
)

render

In [ ]:
result = graph.invoke({"question": VALIDATION_QUESTION})

In [ ]:
for i, doc in enumerate(result["context"]):
    rich_print(f'Contexto {i+1}: {doc.page_content}')
rich_print(f'Resposta: {result["answer"]}')

In [ ]:
# Stream steps
for step in graph.stream(
    {"question": VALIDATION_QUESTION},
    stream_mode="updates",
):
    rich_print(f"{step}\n\n----------------\n")

In [ ]:
# Stream tokens
for message, metadata in graph.stream(
    {"question": VALIDATION_QUESTION},
    stream_mode="messages",
):
    print(message.content, end="|")